# 功能调试

Pro Kernel 出问题，先**固定可复现条件**（输入 Shape、dtype、TilingKey、block_dim），再按现象走三条路径之一。


## 1. 路径一：编译报错

JIT 生成新实例时依次经过参数绑定、前端解析校验、代码生成、编译——先看异常中的**用户源码位置、错误描述、调用栈**判断失败环节：

- **参数绑定 / 前端解析 / IR 校验失败**：重点检查接口参数、Shape、数据类型、内存空间与编译期约束；带 PyPTO 错误码的，查错误码参考所属组件；
- **代码生成 / 工具链编译失败**：查完整编译日志与生成代码。

JIT 产物默认在 `build` 目录（设置 `ASCEND_WORK_PATH` 后在 `${ASCEND_WORK_PATH}/PYPTO_PRO/build`）：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">文件</th><th style="text-align:left;">用途</th></tr>
<tr><td><code>kernel.cpp</code></td><td>编译器生成的设备侧源码：检查参数类型、控制流、地址计算、搬运</td></tr>
<tr><td><code>pipeline_generated.py</code></td><td>自动流水生成的核间排布代码：检查 stage/preload/同步</td></tr>
</table>

**生成文件只读不改**——定位到问题后改原始 Python Kernel 再重编译。


## 2. 路径二：精度错误

Kernel 正常执行但结果不对：

1. 先核对一致性：输入、Shape、dtype、布局、比较阈值；确认在 `torch.npu.synchronize()` **之后**读结果；
2. 固定失败用例，沿**实际数据路径**找第一个出错环节；
3. 需要看标量信息（Core 编号、偏移、Shape、分支标志）时用 `pypto_pro.language.printf`——多核场景用 `get_block_idx()` 限制打印单个 Core，避免日志交错；该接口有运行时开销，**仅用于功能调试**；
4. 结果错乱/偶发错/疑似内存问题 → 开 sanitizer（下一节）。


## 3. 路径三：AI Core Error

越界读写、死锁超时等设备侧异常。要点：

1. Kernel 下发是异步的——**同步 Stream 让异常在调用点暴露**：`kernel[None, block_dim](*args); torch.npu.synchronize()`，保留完整报错与日志；
2. 设置可写工作目录 `export ASCEND_WORK_PATH=./wk`；确保未设置 `NPU_COLLECT_PATH`（有则 unset），否则无法生成离线复现数据；建议在**独立子进程**跑问题用例；
3. 异常后产物保存在 `${ASCEND_WORK_PATH}/extra-info/data-dump/<device_id>/`：dump 数据文件（输入/输出/workspace 原始数据）、`<kernel>_launch_args.json`（启动参数）、`<kernel>_call_kernel.so`（带调试信息的执行体，用于离线复现与 **Error PC 定位到源码行**）。


## 4. 日志、错误码与数据 Dump

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">工具</th><th style="text-align:left;">用途</th><th style="text-align:left;">位置</th></tr>
<tr><td>前端异常与错误码</td><td>判断失败环节（参数绑定/解析/IR 校验），结合错误码参考定位组件</td><td>异常信息内联</td></tr>
<tr><td>编译产物日志</td><td>代码生成/工具链失败的完整日志</td><td><code>${ASCEND_WORK_PATH}/PYPTO_PRO/build</code></td></tr>
<tr><td><code>pypto_pro.language.printf</code></td><td>Kernel 内标量打印（Core 号、偏移、Shape、分支标志）</td><td>仅功能调试</td></tr>
<tr><td><code>pl.dump_data</code></td><td>按窗口打印 Pro Tensor/Tile 内容（按 API 限制使用），属于当前源码的调试入口</td><td>终端输出，与 printf 共享 FIFO；目标发行包能力待核对</td></tr>
</table>

**排错工作流建议**：报错 → 先看异常中的用户源码位置（最快路径）→ 带错误码查参考 → 仍不明再看 build 产物与日志。


## 5. 案例实战（理论推导版）

**案例：偶发精度错误——Tile 地址重叠**

- **现象**：Kernel 小规模正确，增大 shape 后输出偶发错乱，无报错，重跑结果不稳定；
- **定位思路**：偶发 + 无报错 → 优先怀疑内存访问类问题（第 7 章 Top5 清单）→ 开 `sanitizer=True` 重跑；
- **根因**：shape 变大后单 Tile 字节数增加（如 [8,256]→[16,512] FP32 从 8KB 变 32KB），原有 `addrs=0x0000/0x2000/0x4000` 的 8KB 间隔布局发生重叠，两组 Tile 互相覆写（时序不同 → 结果偶发）；
- **修复**：按新字节数重排地址（0x0000/0x8000/0x10000），并回归原失败 shape。

【待 950 验证：案例为课程构造（依据 sanitizer 检测范围与第 2 章地址规则推导），可用于教学复现】


## 6. 修复后的收尾纪律

关闭 sanitizer、移除调试 printf、生产配置重新编译，**覆盖原失败用例及相关 Shape/dtype/TilingKey/block_dim 做回归**——问题才算关闭。


## 7. 三路径的"第一反应"清单

把本节内容压缩成遇到问题时的条件反射（贴在手边）：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">现象</th><th style="text-align:left;">第一反应</th><th style="text-align:left;">第二步</th></tr>
<tr><td>首次调用就报错（Python 侧异常）</td><td><b>路径一</b>：看异常里的源码行号</td><td>有错误码 → 查参考；否则检查类型标注/shape/接口参数</td></tr>
<tr><td>跑通了但数值不对</td><td>确认 <code>synchronize()</code> 后再读结果</td><td>固定失败用例 → printf 打印标量 → 疑内存问题开 sanitizer</td></tr>
<tr><td>Kernel 卡死 / AI Core Error</td><td>同步暴露完整报错</td><td>设 ASCEND_WORK_PATH 采 dump → 离线复现定位 Error PC</td></tr>
<tr><td>偶发错、时对时错</td><td>开 sanitizer</td><td>检出的越界/重叠直接指向问题 Tile</td></tr>
</table>

**一个重要习惯**：修完一个问题后，**把失败用例留成回归脚本**（固定 seed + shape + 断言），并入你的算子验收集——"这次修好了"和"以后不会再坏"是两件事。


### 把“偶发错误”缩成可以核对的证据

先保存输入 shape/dtype、seed、启动核数和失败索引。第 1 章由 [8,256] 扩为 [16,512] FP32 后，每块应占 0x8000 B；原来 0x2000 的地址间隔会让三组区间两两相交。[地址算例脚本](./src/address_case.py) 用纯整数计算展示修复前后的区间，不执行有缺陷的设备 Kernel。

精度检查要记录第一个不匹配元素的坐标、实际值和 Golden，以及最大绝对误差；NaN 也应算失败。后面的 [统一实验脚本](./src/benchmark_fusion.py) 提供 `first_mismatch`，可以把“通过/不通过”进一步定位到数据区域。真实错误日志需在 950 实测后附上，教材不预填假日志。


## 8. 课后练习

1. （填空）精度对比必须在 `______` 之后读结果；理由是 ______。
2. （选择）怀疑 GM 访问越界导致偶发错误，第一步：
   A. 直接改代码<br/>B. 开 sanitizer<br/>C. 换输入重跑
3. （实践）为你的 RMSNorm 实践写一个"最小失败用例"脚本：固定 seed、固定 shape、失败即打印首个不匹配位置的行号列号。

In [ ]:
!cat ./answer/08.02_answer.txt

## 9. 本节小结

- 三路径：**编译报错（看源码位置与 build 产物）/ 精度错误（printf 标量）/ AI Core Error（同步暴露 + 异常 dump 复现）**；
- build 产物只读；printf 只用于调试；
- 修复后必须回归——纪律与工具同样重要。